## Question 1: [Index] S&P 500 Stocks Added to the Index

### Which year had the highest number of additions (starting from 2020)?

In [73]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime as dt


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# send request to url
response = requests.get(url, headers=headers)
response.raise_for_status()

# get table
soup = BeautifulSoup(response.content, "html.parser")
table = soup.find("table", id="constituents")

df = pd.read_html(str(table))[0]

# Extract year added from Date added
df["Date added"] = pd.to_datetime(df["Date added"])
df["year_added"] = df["Date added"].dt.year

# group by year added and count companies added in that year
df_yearly_add = df[["year_added", "Symbol"]].groupby(
    'year_added'
  ).count().reset_index()

# filter to 2020 and above only
df_filtered = df_yearly_add[df_yearly_add.year_added >= 2020]
display(df_filtered)
print(df_filtered[df_filtered.Symbol == max(df_filtered.Symbol)])

/tmp/ipykernel_8414/1060976406.py:20: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]


,year_added,Symbol
51,2020,10
52,2021,10
53,2022,15
54,2023,15
55,2024,16
56,2025,18
57,2026,13


    year_added  Symbol
56        2025      18


Answer: From 2020 onwards, The year with the most companies added is 2025 with 18 additions.

### Additional: How many current S&P 500 stocks have been in the index for more than 20 years?

In [84]:
df_old_companies = df[df.year_added < dt.today().year - 20]
display(df_old_companies.head())
display(df_old_companies.shape[0])

### double check:
# ((dt.now().year - df.year_added) > 20).sum()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,year_added
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,1957
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,1957
5,ADBE,Adobe Inc.,Information Technology,Application Software,"San Jose, California",1997-05-05,796343,1982,1997
7,AES,AES Corporation,Utilities,Independent Power Producers & Energy Traders,"Arlington, Virginia",1998-10-02,874761,1981,1998
8,AFL,Aflac,Financials,Life & Health Insurance,"Columbus, Georgia",1999-05-28,4977,1955,1999


218

Answer: There are 218 companies in the S&P500 that has been there for at least 20 years.

## Question 2. [Macro] Indexes YTD (as of 21 August 2026)

### How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?

In [145]:
import datetime as dt
import yfinance as yf
from time import sleep

# build indices dict to loop through
indices = [
    {"index": "^GSPC", "country": "US"},
    {"index": "000001.SS", "country": "China"},
    {"index": "^HSI", "country": "Hong Konf"},
    {"index": "^AXJO", "country": "Australia"},
    {"index": "^NSEI", "country": "India"},
    {"index": "^GSPTSE", "country": "Canada"},
    {"index": "^GDAXI", "country": "Germany"},
    {"index": "^FTSE", "country": "UK"},
    {"index": "^N225", "country": "Japan"},
    {"index": "^MXX", "country": "Mexico"},
    {"index": "^BVSP", "country": "Brazil"}
]
start = dt.date(year=2026, month=1, day=1)
end = dt.date(year=2026, month=8, day=21)

# get data then calculate YTD growth and save to dict
for i in indices:
  print(f"Getting data for {i["country"]}.")
  tempdf = yf.Ticker(i["index"]).history(start=start, end=end)
  tempdf['ytd_growth'] = tempdf.Close / tempdf.Close.shift(
    tempdf.shape[0] - 1
  ) - 1
  i["ytd_growth"] = tempdf[-1:].ytd_growth[0]
  sleep(2)

# convert dict to DataFrame
df = pd.DataFrame(indices)

# find countries with better YTD growth than the US
display(df[df.ytd_growth > df[df.country == "US"].ytd_growth.iloc[0]])

Getting data for US.


/tmp/ipykernel_8414/4261642315.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  i["ytd_growth"] = tempdf[-1:].ytd_growth[0]


Getting data for China.


/tmp/ipykernel_8414/4261642315.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  i["ytd_growth"] = tempdf[-1:].ytd_growth[0]


Getting data for Hong Konf.


...

Getting data for Mexico.


/tmp/ipykernel_8414/4261642315.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  i["ytd_growth"] = tempdf[-1:].ytd_growth[0]


Getting data for Brazil.


/tmp/ipykernel_8414/4261642315.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  i["ytd_growth"] = tempdf[-1:].ytd_growth[0]


,index,country,ytd_growth
5,^GSPTSE,Canada,0.140575
8,^N225,Japan,0.277507


Answer: There are 2 countries performing better, whihc are Canada and Japan.

## Question 3. [Index] S&P 500 Market Corrections Analysis

### Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.

In [57]:
import yfinance as yf
import datetime as dt


start = dt.date(year=1950, month=1, day=1)
end = dt.date.today()

# df = yf.Ticker("^GSPC").history(start=start, end=end)
df = df[['Close']]

# df["previous_day_close"] = df.Close.shift(1)
# df["closed_high"] = df.Close > df.previous_day_close

# Calculate the all-time high closing price up to each day
df['all_time_high_close'] = df['Close'].expanding().max()
# Identify days where the Close price is an all-time high
df['is_all_time_high'] = (df['Close'] == df['all_time_high_close'])

df["consecutive_ath"] = df.is_all_time_high & df.is_all_time_high.shift(1)
df["prev_ath"] = df.all_time_high_close.shift(1)

df.head(20)

,Close,all_time_high_close,is_all_time_high,consecutive_ath,prev_ath
Date,,,,,
1950-01-03 00:00:00-05:00,16.660000,16.66,True,False,NaN
1950-01-04 00:00:00-05:00,16.850000,16.85,True,True,16.66
1950-01-05 00:00:00-05:00,16.930000,16.93,True,True,16.85
1950-01-06 00:00:00-05:00,16.980000,16.98,True,True,16.93
1950-01-09 00:00:00-05:00,17.080000,17.08,True,True,16.98
1950-01-10 00:00:00-05:00,17.030001,17.08,False,False,17.08
1950-01-11 00:00:00-05:00,17.090000,17.09,True,False,17.08
1950-01-12 00:00:00-05:00,16.760000,17.09,False,False,17.09
1950-01-13 00:00:00-05:00,16.670000,17.09,False,False,17.09


In [26]:
import pandas as pd

# Get the dates of all all-time high closing prices
ath_dates = df[df['is_all_time_high']].index.tolist()

min_prices_between_aths = []

# Iterate through consecutive pairs of all-time high dates
for i in range(len(ath_dates) - 1):
    start_ath_date = ath_dates[i]
    end_ath_date = ath_dates[i+1]

    # Get the slice of the DataFrame strictly between the two all-time highs
    # We need to find the index positions to correctly slice (exclusive of ATHs)
    idx_start = df.index.get_loc(start_ath_date)
    idx_end = df.index.get_loc(end_ath_date)

    # Ensure there are days between the two ATHs to calculate a minimum
    if idx_end > idx_start + 1:
        # Slice the 'Close' prices strictly between the two ATHs
        intermediate_prices = df.iloc[idx_start + 1 : idx_end]['Close']

        if not intermediate_prices.empty:
            min_price = intermediate_prices.min()
            min_prices_between_aths.append({
                'start_ath_date': start_ath_date,
                'end_ath_date': end_ath_date,
                'min_price_between': min_price
            })

# Convert the list of dictionaries to a DataFrame for easier viewing
df_min_between_aths = pd.DataFrame(min_prices_between_aths)
display(df_min_between_aths.head())

,is_all_time_high
Date,
1950-01-03 00:00:00-05:00,False
1950-01-04 00:00:00-05:00,True
1950-01-05 00:00:00-05:00,True
1950-01-06 00:00:00-05:00,True
1950-01-09 00:00:00-05:00,True
...,...
2026-09-04 00:00:00-04:00,False
2026-09-08 00:00:00-04:00,False
2026-09-09 00:00:00-04:00,False


In [161]:
# Example of 'reverse expanding' to find the 'future all-time high'
# This calculates, for each day, the maximum closing price observed from that day until the end of the dataset.
df['future_all_time_high_close'] = df['Close'].iloc[::-1].expanding().max().iloc[::-1]

# Display relevant columns to see the effect
# display(df[['Close', 'all_time_high_close', 'future_all_time_high_close']].head())
display(df)

In [21]:
df.tail(20)

,Close,all_time_high_close,is_all_time_high
Date,,,
2026-08-14 00:00:00-04:00,7785.759766,7798.990234,False
2026-08-17 00:00:00-04:00,7745.060059,7798.990234,False
2026-08-18 00:00:00-04:00,7691.759766,7798.990234,False
2026-08-19 00:00:00-04:00,7707.979980,7798.990234,False
2026-08-20 00:00:00-04:00,7641.160156,7798.990234,False
2026-08-21 00:00:00-04:00,7674.370117,7798.990234,False
2026-08-24 00:00:00-04:00,7652.859863,7798.990234,False
2026-08-25 00:00:00-04:00,7677.279785,7798.990234,False
2026-08-26 00:00:00-04:00,7675.700195,7798.990234,False
